In [1]:
!pip install wbgapi scikit-learn statsmodels matplotlib seaborn numpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 548.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 7.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


In [ ]:
# Required libraries: imports and configuration
import wbgapi as wb
import pandas as pd
import numpy as np
from sklearn.linear_model import (
    LinearRegression, RidgeCV, LassoCV, lasso_path, LogisticRegression
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    precision_recall_curve, f1_score, precision_score, recall_score
)
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# ── Indicator dictionary (25+ WDI indicators) ──
indicators = {
    "NY.GDP.PCAP.KD.ZG": "gdp_growth_pc",
    "NY.GDP.PCAP.KD":     "gdp_per_capita",
    "FP.CPI.TOTL.ZG":     "inflation",
    "NE.TRD.GNFS.ZS":     "trade_pct_gdp",
    "GC.DOD.TOTL.GD.ZS":  "govt_debt_pct_gdp",
    "GC.REV.XGRT.GD.ZS":  "govt_revenue_pct_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_pct_gdp",
    "BN.CAB.XOKA.GD.ZS":  "current_account_pct_gdp",
    "FR.INR.RINR":         "real_interest_rate",
    "SL.UEM.TOTL.ZS":     "unemployment_rate",
    "SP.POP.TOTL":         "population",
    "SP.POP.GROW":         "pop_growth",
    "SE.XPD.TOTL.GD.ZS":  "edu_expenditure_pct_gdp",
    "SH.XPD.CHEX.GD.ZS":  "health_expenditure_pct_gdp",
    "EG.USE.PCAP.KG.OE":  "energy_use_per_capita",
    "IT.NET.USER.ZS":      "internet_users_pct",
    "SP.URB.TOTL.IN.ZS":  "urban_population_pct",
    "AG.LND.ARBL.ZS":     "arable_land_pct",
    "MS.MIL.XPND.GD.ZS":  "military_expenditure_pct_gdp",
    "IC.BUS.EASE.XQ":      "ease_of_doing_business",
    "TX.VAL.TECH.MF.ZS":  "high_tech_exports_pct",
    "NV.IND.MANF.ZS":     "manufacturing_pct_gdp",
    "NV.AGR.TOTL.ZS":     "agriculture_pct_gdp",
    "NV.SRV.TOTL.ZS":     "services_pct_gdp",
    "FB.BNK.CAPA.ZS":     "bank_capital_ratio",
    "FS.AST.DOMS.GD.ZS":  "domestic_credit_pct_gdp",
    "CM.MKT.LCAP.GD.ZS":  "market_cap_pct_gdp",
    "PA.NUS.FCRF":         "exchange_rate",
    "GC.XPN.TOTL.GD.ZS":  "govt_expenditure_pct_gdp",
    "SP.DYN.LE00.IN":      "life_expectancy",
}

# ── Step 1: Download from World Bank API (2013–2019) ──
print("Downloading WDI indicators from World Bank API...")
raw = wb.data.DataFrame(
    list(indicators.keys()),
    time=range(2013, 2020),
    labels=False,
    numericTimeKeys=True
).reset_index()

# Rename columns
raw = raw.rename(columns=indicators)

# ── Step 2: Collapse to country-level means ──
# Group by country, average across years
country_means = raw.groupby("economy").mean(numeric_only=True).reset_index()
country_means = country_means.rename(columns={"economy": "country"})

n_countries_raw = len(country_means)
n_indicators_raw = len(country_means.columns) - 1

# Drop countries missing >40% of indicators
threshold_countries = 0.40 * n_indicators_raw
country_means = country_means.dropna(thresh=int(n_indicators_raw - threshold_countries + 1))

# Drop indicators missing >40% of countries
threshold_indicators = 0.40 * len(country_means)
country_means = country_means.dropna(axis=1, thresh=int(len(country_means) - threshold_indicators + 1))

# Median-impute remaining gaps
for col in country_means.select_dtypes(include=np.number).columns:
    country_means[col] = country_means[col].fillna(country_means[col].median())

# ── Step 3: Define continuous outcome ──
# gdp_growth_pc already in dataset as the average growth rate

# ── Step 4: Define binary crisis outcome ──
country_means["crisis"] = (country_means["gdp_growth_pc"] < 0).astype(int)

# ── Step 5: Train/test split + standardize ──
feature_cols = [c for c in country_means.columns if c not in ["country", "gdp_growth_pc", "crisis"]]
X = country_means[feature_cols]
y_binary = country_means["crisis"]
y_continuous = country_means["gdp_growth_pc"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.30, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Convert back to DataFrames for convenience
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_cols, index=X_test.index)

# ── Print required summary stats ──
n_total     = len(country_means)
n_crisis    = int(y_binary.sum())
n_no_crisis = n_total - n_crisis
base_rate   = n_crisis / n_total

print(f"\n── Dataset Dimensions ──")
print(f"Countries:  {n_total}")
print(f"Features:   {len(feature_cols)}")
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

print(f"\n── Crisis Distribution ──")
print(f"Crisis     (1): {n_crisis}")
print(f"No Crisis  (0): {n_no_crisis}")
print(f"Base rate:      {base_rate:.3f} ({base_rate*100:.1f}%)")

/opt/anaconda3/lib/python3.12/site-packages/wbgapi/__init__.py:639: SyntaxWarning: invalid escape sequence '\w'
  pattern = '(?<!\w).{{0,{len}}}{term}.{{0,{len}}}(?!\w)'.format(term=re.escape(q), len=padding)


SSLError: HTTPSConnectionPool(host='api.worldbank.org', port=443): Max retries exceeded with url: /v2/en/sources/2/series/NY.GDP.PCAP.KD.ZG;NY.GDP.PCAP.KD;FP.CPI.TOTL.ZG;NE.TRD.GNFS.ZS;GC.DOD.TOTL.GD.ZS;GC.REV.XGRT.GD.ZS;BX.KLT.DINV.WD.GD.ZS;BN.CAB.XOKA.GD.ZS;FR.INR.RINR;SL.UEM.TOTL.ZS;SP.POP.TOTL;SP.POP.GROW;SE.XPD.TOTL.GD.ZS;SH.XPD.CHEX.GD.ZS;EG.USE.PCAP.KG.OE;IT.NET.USER.ZS;SP.URB.TOTL.IN.ZS;AG.LND.ARBL.ZS;MS.MIL.XPND.GD.ZS;IC.BUS.EASE.XQ;TX.VAL.TECH.MF.ZS;NV.IND.MANF.ZS;NV.AGR.TOTL.ZS;NV.SRV.TOTL.ZS;FB.BNK.CAPA.ZS;FS.AST.DOMS.GD.ZS;CM.MKT.LCAP.GD.ZS;PA.NUS.FCRF;GC.XPN.TOTL.GD.ZS;SP.DYN.LE00.IN/country/all/time/YR2013;YR2014;YR2015;YR2016;YR2017;YR2018;YR2019?per_page=1000&page=1&format=json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))

In [8]:
# ── Indicator dictionary (25+ WDI indicators) ──
indicators = {
    "NY.GDP.PCAP.KD.ZG": "gdp_growth_pc",
    "NY.GDP.PCAP.KD":     "gdp_per_capita",
    "FP.CPI.TOTL.ZG":     "inflation",
    "NE.TRD.GNFS.ZS":     "trade_pct_gdp",
    "GC.DOD.TOTL.GD.ZS":  "govt_debt_pct_gdp",
    "GC.REV.XGRT.GD.ZS":  "govt_revenue_pct_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_pct_gdp",
    "BN.CAB.XOKA.GD.ZS":  "current_account_pct_gdp",
    "FR.INR.RINR":         "real_interest_rate",
    "SL.UEM.TOTL.ZS":     "unemployment_rate",
    "SP.POP.TOTL":         "population",
    "SP.POP.GROW":         "pop_growth",
    "SE.XPD.TOTL.GD.ZS":  "edu_expenditure_pct_gdp",
    "SH.XPD.CHEX.GD.ZS":  "health_expenditure_pct_gdp",
    "EG.USE.PCAP.KG.OE":  "energy_use_per_capita",
    "IT.NET.USER.ZS":      "internet_users_pct",
    "SP.URB.TOTL.IN.ZS":  "urban_population_pct",
    "AG.LND.ARBL.ZS":     "arable_land_pct",
    "MS.MIL.XPND.GD.ZS":  "military_expenditure_pct_gdp",
    "IC.BUS.EASE.XQ":      "ease_of_doing_business",
    "TX.VAL.TECH.MF.ZS":  "high_tech_exports_pct",
    "NV.IND.MANF.ZS":     "manufacturing_pct_gdp",
    "NV.AGR.TOTL.ZS":     "agriculture_pct_gdp",
    "NV.SRV.TOTL.ZS":     "services_pct_gdp",
    "FB.BNK.CAPA.ZS":     "bank_capital_ratio",
    "FS.AST.DOMS.GD.ZS":  "domestic_credit_pct_gdp",
    "CM.MKT.LCAP.GD.ZS":  "market_cap_pct_gdp",
    "PA.NUS.FCRF":         "exchange_rate",
    "GC.XPN.TOTL.GD.ZS":  "govt_expenditure_pct_gdp",
    "SP.DYN.LE00.IN":      "life_expectancy",
}

# ── Step 1: Download from World Bank API (2013–2019) ──
print("Downloading WDI indicators from World Bank API...")
raw = wb.data.DataFrame(
    list(indicators.keys()),
    time=range(2013, 2020),
    labels=False,
    numericTimeKeys=True
).reset_index()

# Rename columns
raw = raw.rename(columns=indicators)

# ── Step 2: Collapse to country-level means ──
# Group by country, average across years
country_means = raw.groupby("economy").mean(numeric_only=True).reset_index()
country_means = country_means.rename(columns={"economy": "country"})

n_countries_raw = len(country_means)
n_indicators_raw = len(country_means.columns) - 1

# Drop countries missing >40% of indicators
threshold_countries = 0.40 * n_indicators_raw
country_means = country_means.dropna(thresh=int(n_indicators_raw - threshold_countries + 1))

# Drop indicators missing >40% of countries
threshold_indicators = 0.40 * len(country_means)
country_means = country_means.dropna(axis=1, thresh=int(len(country_means) - threshold_indicators + 1))

# Median-impute remaining gaps
for col in country_means.select_dtypes(include=np.number).columns:
    country_means[col] = country_means[col].fillna(country_means[col].median())

# ── Step 3: Define continuous outcome ──
# gdp_growth_pc already in dataset as the average growth rate

# ── Step 4: Define binary crisis outcome ──
country_means["crisis"] = (country_means["gdp_growth_pc"] < 0).astype(int)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:

# ── Step 5: Train/test split + standardize ──
feature_cols = [c for c in country_means.columns if c not in ["country", "gdp_growth_pc", "crisis"]]
X = country_means[feature_cols]
y_binary = country_means["crisis"]
y_continuous = country_means["gdp_growth_pc"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.30, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Convert back to DataFrames for convenience
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_cols, index=X_test.index)

# ── Print required summary stats ──
n_total     = len(country_means)
n_crisis    = int(y_binary.sum())
n_no_crisis = n_total - n_crisis
base_rate   = n_crisis / n_total

print(f"\n── Dataset Dimensions ──")
print(f"Countries:  {n_total}")
print(f"Features:   {len(feature_cols)}")
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

print(f"\n── Crisis Distribution ──")
print(f"Crisis     (1): {n_crisis}")
print(f"No Crisis  (0): {n_no_crisis}")
print(f"Base rate:      {base_rate:.3f} ({base_rate*100:.1f}%)")